In [0]:
#%run ./env ----- A decommenter pour lancer le notebook separement
#%run ./python_libraries ----- A decommenter pour lancer le notebook separement
#%run ../delta_function ----- A decommenter pour lancer le notebook separement
#%run ./load_data ----- A decommenter pour lancer le notebook separement
#%run ./transform_data ----- A decommenter pour lancer le notebook separement

In [0]:
#%run ./env

In [0]:
#%run ./python_libraries 

In [0]:
#%run ../delta_function 

In [0]:
#%run ./load_data 

In [0]:
#%run ./transform_data 

## Construction dim_batch
Une ligne = un batch. Cle primaire = batch_id.

In [0]:
dim_batch = (
    batches
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_batch").alias("batch_id"),
        F.col("batch_number"),
        F.col("requirement_specifications"),
        F.col("variety"),
        F.col("fabrication_order_number"),
        F.col("mes_number"),
        F.col("planned_date"),
        F.col("harvest"),
        F.col("production_line"),
        F.col("production_type"),
        F.col("status"),
        F.col("deleted"),
        F.col("batch_cycle"),
        F.col("planned_time"),
        F.col("planned_datetime"),
        F.col("created_at"),
        F.col("updated_at"),
        F.col("deleted_at")
    )
)

# production_line reste en FK brute (int) : la relation vers dim_site
# (id_plant_production_line) se fait cote Power BI, pas de jointure ici.

# max_end_date_kiln_unload : attribut du batch, calcule une fois, vraie Date.
# Regroupement par batch_id uniquement (pas prd_line) : evite les doublons
# lies a d'eventuelles variations de prd_line pour un meme batch.
max_end_date_kiln_unload = (
    localization_events_union
    .filter(
        (F.col("localization_event") == "kiln_unload") &
        (F.col("batch_id").isNotNull())
    )
    .groupBy("batch_id")
    .agg(F.max("end").alias("max_end_date_kiln_unload_brute"))
    .withColumn("max_end_date_kiln_unload", F.to_date(F.col("max_end_date_kiln_unload_brute")))
    .select("batch_id", "max_end_date_kiln_unload")
)

dim_batch = (
    dim_batch.alias("a")
    .join(
        max_end_date_kiln_unload.alias("b"),
        F.col("a.batch_id") == F.col("b.batch_id"),
        "left"
    )
    .select(
        "a.*",
        "b.max_end_date_kiln_unload"
    )
)

# variety et requirement_specifications sont des FK : on rapatrie le libelle
# depuis goods_varieties et requirement_specifications. La FK est conservee sous
# variety_id / specification_id, le libelle reprend le nom d'origine pour ne rien
# changer cote Power BI.
dim_batch = (
    dim_batch.alias("a")
    .join(
        goods_varieties.select(
            F.col("id_good_variety"),
            F.col("code").alias("variety_label")
        ).alias("v"),
        F.col("a.variety") == F.col("v.id_good_variety"),
        "left"
    )
    .join(
        requirement_specifications.select(
            F.col("id_requirement_specification"),
            F.col("name").alias("specification_label")
        ).alias("s"),
        F.col("a.requirement_specifications") == F.col("s.id_requirement_specification"),
        "left"
    )
    .select(
        "a.*",
        "v.variety_label",
        "s.specification_label"
    )
)

dim_batch = (
    dim_batch
    .withColumnRenamed("variety", "variety_id")
    .withColumnRenamed("requirement_specifications", "specification_id")
    .withColumnRenamed("variety_label", "variety")
    .withColumnRenamed("specification_label", "requirement_specifications")
)


## Ecriture Delta

In [0]:
current_process = "dim_batch"
target_dim_batch = current_catalog + "." + current_schema + "." + current_process
print(target_dim_batch)

In [0]:
all_columns = dim_batch.columns
primary_key = ['batch_id']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

In [0]:
handle_table_update(
    dim_batch,
    target_dim_batch,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode
)